# Global EV Transition Readiness Index

The aim of this project is to build a Global EV Transition Readiness Index.

The index combines EV adoption, charging infrastructure, national readiness, and electricity sustainability indicators into one country-level score. The work starts by loading and cleaning the source datasets, dealing with missing values, checking outliers, and preparing a complete dataset for the index calculations.

## Import Libraries

In [ ]:
# Import the libraries needed for cleaning, checking, and plotting the data.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Load Datasets

The project uses EV data from the IEA, population and national context data from the World Bank, low-carbon electricity data from OWID, and scraped renewable electricity data from CountryEconomy.

In [ ]:
# Load all source datasets from the Datasets folder.
df_iea = pd.read_excel(r"Datasets\EVDataExplorer2025.xlsx", sheet_name="GEVO_EV_2025")
df_population = pd.read_csv(r"Datasets\API_SP\API_SP.POP.TOTL_DS2_en_csv_v2_58.csv", skiprows=4)
df_gdp = pd.read_csv(r"Datasets\API_NY\API_NY.GDP.PCAP.PP.KD_DS2_en_csv_v2_1700.csv", skiprows=4)
df_electricity = pd.read_csv(r"Datasets\API_EG\API_EG.USE.ELEC.KH.PC_DS2_en_csv_v2_1655.csv", skiprows=4)
df_owid = pd.read_csv(r"Datasets\owid_share_electricity_low_carbon_full.csv")
df_renewable = pd.read_csv(r"Datasets\countryeconomy_renewable_electricity_2024.csv")

## Preview Source Data

In [ ]:
# Preview the IEA EV data to check it loaded correctly.
df_iea.head()

In [ ]:
# Preview the population data.
df_population.head()

In [ ]:
# Preview the GDP per capita data.
df_gdp.head()

In [ ]:
# Preview the electric power consumption data.
df_electricity.head()

In [ ]:
# Preview the OWID low-carbon electricity data.
df_owid.head()

In [ ]:
# Preview the renewable electricity dataset.
df_renewable.head()

## Check Dataset Size and Structure

In [ ]:
# Print the size of each dataset so the inputs are documented.
print("IEA EV data:", df_iea.shape)
print("Population data:", df_population.shape)
print("GDP data:", df_gdp.shape)
print("Electricity consumption data:", df_electricity.shape)
print("OWID low-carbon electricity data:", df_owid.shape)
print("Scraped renewable electricity data:", df_renewable.shape)

In [ ]:
# Check the columns and data types in the IEA file.
df_iea.info()

In [ ]:
# Check the columns and data types in the population file.
df_population.info()

In [ ]:
# Check the columns and data types in the GDP per capita file.
df_gdp.info()

In [ ]:
# Check the columns and data types in the electric power consumption file.
df_electricity.info()

In [ ]:
# Check the columns and data types in the OWID low-carbon electricity file.
df_owid.info()

In [ ]:
# Check the columns and data types in the renewable electricity file.
df_renewable.info()

## Clean IEA EV Data

The IEA workbook contains different years, vehicle modes, powertrains, and both historical and projected values. Only historical 2024 country data is selected for the index.

In [ ]:
# Keep the 2024 historical EV records needed for the index.
df_iea_2024 = df_iea[df_iea["year"] == 2024].copy()
df_iea_2024 = df_iea_2024[df_iea_2024["category"] == "Historical"]

# Car adoption indicators and public charger indicators are selected separately.
df_iea_2024 = df_iea_2024[
    ((df_iea_2024["parameter"].isin(["EV sales share", "EV stock share"])) &
     (df_iea_2024["mode"] == "Cars") &
     (df_iea_2024["powertrain"] == "EV")) |
    ((df_iea_2024["parameter"] == "EV stock") &
     (df_iea_2024["mode"] == "Cars") &
     (df_iea_2024["powertrain"].isin(["BEV", "PHEV", "FCEV"]))) |
    ((df_iea_2024["parameter"] == "EV charging points") &
     (df_iea_2024["mode"] == "EV") &
     (df_iea_2024["powertrain"].isin(["Publicly available slow", "Publicly available fast"])))
]

df_iea_2024.head()

In [ ]:
# Check that the selected IEA indicators are the ones needed.
df_iea_2024[["parameter", "mode", "powertrain"]].value_counts()

In [ ]:
# EV sales share for cars
df_sales_share = df_iea_2024[
    (df_iea_2024["parameter"] == "EV sales share") &
    (df_iea_2024["mode"] == "Cars")
][["region_country", "value"]].copy()
df_sales_share = df_sales_share.rename(columns={"value": "EV_sales_share"})

# EV stock share for cars
df_stock_share = df_iea_2024[
    (df_iea_2024["parameter"] == "EV stock share") &
    (df_iea_2024["mode"] == "Cars")
][["region_country", "value"]].copy()
df_stock_share = df_stock_share.rename(columns={"value": "EV_stock_share"})

# EV stock is split by powertrain, so BEV, PHEV, and FCEV are summed.
df_stock = df_iea_2024[
    (df_iea_2024["parameter"] == "EV stock") &
    (df_iea_2024["mode"] == "Cars")
].groupby("region_country", as_index=False)["value"].sum()
df_stock = df_stock.rename(columns={"value": "EV_stock"})

# Public slow charging points
df_slow = df_iea_2024[
    (df_iea_2024["parameter"] == "EV charging points") &
    (df_iea_2024["powertrain"] == "Publicly available slow")
][["region_country", "value"]].copy()
df_slow = df_slow.rename(columns={"value": "Public_slow_chargers"})

# Public fast charging points
df_fast = df_iea_2024[
    (df_iea_2024["parameter"] == "EV charging points") &
    (df_iea_2024["powertrain"] == "Publicly available fast")
][["region_country", "value"]].copy()
df_fast = df_fast.rename(columns={"value": "Public_fast_chargers"})

In [ ]:
# Merge the selected IEA indicators into one country-level EV table.
df_iea_clean = df_sales_share.copy()
df_iea_clean = pd.merge(df_iea_clean, df_stock_share, on="region_country", how="outer")
df_iea_clean = pd.merge(df_iea_clean, df_stock, on="region_country", how="outer")
df_iea_clean = pd.merge(df_iea_clean, df_slow, on="region_country", how="outer")
df_iea_clean = pd.merge(df_iea_clean, df_fast, on="region_country", how="outer")

df_iea_clean = df_iea_clean.rename(columns={"region_country": "Country"})

df_iea_clean.head()